In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cvxpy as cp
import qnt.data as qndata

# ============================================================
# 1) Load Quantiacs data + pick random liquid assets
# ============================================================
def load_quantiacs_universe(seed=7, n_assets=50):
    data = qndata.stocks.load_data()  # DataArray(field, time, asset)
    close = data.sel(field="close")
    is_liquid = data.sel(field="is_liquid")

    liquid_last = is_liquid.isel(time=-1).fillna(0).astype(bool)
    liquid_assets = close.asset.values[liquid_last.values]

    if len(liquid_assets) < n_assets:
        raise ValueError(f"Not enough liquid assets ({len(liquid_assets)}) to pick {n_assets}.")

    rng = np.random.default_rng(seed)
    chosen_assets = rng.choice(liquid_assets, size=n_assets, replace=False)
    return data, chosen_assets


# ============================================================
# 2) Returns matrix R (daily) from close
# ============================================================
def compute_returns_matrix(data, chosen_assets, use_log_returns=False):
    close = data.sel(field="close").sel(asset=chosen_assets)

    if use_log_returns:
        rets = np.log(close / close.shift(time=1))
    else:
        rets = close / close.shift(time=1) - 1

    rets = rets.dropna("time")  # DataArray(time, asset)
    times = rets.time.values
    R = rets.values  # (T, n)
    return times, R


# ============================================================
# 3) Rolling Sigma_t (covariance) - annualised
# ============================================================
def compute_sigma_series(R, lookback=252, annualize=252):
    T, n = R.shape
    if T <= lookback + 1:
        raise ValueError(f"Not enough history: T={T}, need > {lookback+1}.")

    Sigma_series = np.zeros((T, n, n))
    for t in range(T):
        start = max(0, t - lookback)
        window = R[start:t+1, :]

        if window.shape[0] > 2:
            Sigma = np.cov(window, rowvar=False)
        else:
            Sigma = np.eye(n) * 1e-6

        Sigma_series[t] = Sigma * annualize

    return Sigma_series


# ============================================================
# 4) Alpha models: simple vs AR(1)
# ============================================================
def mu_series_simple(R, lookback_mu=60, mode="momentum", annualize=252):
    """
    Simple predictor:
      momentum: mean of last K returns
      mean_reversion: negative mean of last K returns
    Uses only data up to time t (inclusive).
    """
    T, n = R.shape
    mu = np.zeros((T, n))
    for t in range(T):
        start = max(0, t - lookback_mu + 1)
        window = R[start:t+1, :]
        m = np.nanmean(window, axis=0)
        if mode == "mean_reversion":
            m = -m
        mu[t] = m * annualize
    return mu


def mu_series_ar1(R, lookback_model=252, annualize=252, smooth_alpha=0.20):
    """
    Rolling AR(1) per asset via OLS:
        r_tau = a + b * r_{tau-1} + eps
    Forecast at time t:
        mu_t = a_hat + b_hat * r_t

    Optional smoothing:
        mu_sm[t] = alpha * mu[t] + (1-alpha) * mu_sm[t-1]
    """
    T, n = R.shape
    mu_raw = np.zeros((T, n))

    for t in range(T):
        start = max(1, t - lookback_model + 1)
        window = R[start:t+1, :]
        if window.shape[0] < 3:
            mu_raw[t] = 0.0
            continue

        X = window[:-1, :]
        Y = window[1:, :]

        for i in range(n):
            x = X[:, i]
            y = Y[:, i]
            A = np.column_stack([np.ones_like(x), x])
            theta, *_ = np.linalg.lstsq(A, y, rcond=None)
            a, b = theta
            mu_raw[t, i] = (a + b * R[t, i]) * annualize

    if smooth_alpha is None or smooth_alpha <= 0.0:
        return mu_raw

    mu_sm = np.zeros_like(mu_raw)
    mu_sm[0] = mu_raw[0]
    for t in range(1, T):
        mu_sm[t] = smooth_alpha * mu_raw[t] + (1.0 - smooth_alpha) * mu_sm[t - 1]
    return mu_sm


# ============================================================
# 5) PSD enforcement
# ============================================================
def make_psd(Sigma, ridge=1e-3, eps=1e-8):
    S = 0.5 * (Sigma + Sigma.T)
    S = S + ridge * np.eye(S.shape[0]) + eps * np.eye(S.shape[0])
    return cp.psd_wrap(cp.Constant(S))


# ============================================================
# 6) Constraints
# ============================================================
def add_constraints(w, long_only=True, w_min=None, w_max=None, fully_invested=True):
    cons = []
    n = w.shape[0]
    if fully_invested:
        cons.append(cp.sum(w) == 1)
    if long_only:
        cons.append(w >= 0)
    if w_min is not None:
        cons.append(w >= w_min * np.ones(n))
    if w_max is not None:
        cons.append(w <= w_max * np.ones(n))
    return cons


# ============================================================
# 7) Baseline one-step optimiser
# ============================================================
def solve_one_step(mu, Sigma, w_prev, gamma=5.0, lam=0.06, w_max=0.10, ridge=1e-3):
    n = len(mu)
    w = cp.Variable(n)

    turnover = cp.norm1(w - w_prev)
    Sigma_psd = make_psd(Sigma, ridge=ridge)

    obj = cp.Maximize(
        mu @ w
        - gamma * cp.quad_form(w, Sigma_psd)
        - lam * turnover
    )

    prob = cp.Problem(obj, add_constraints(w, long_only=True, w_max=w_max))
    prob.solve(solver=cp.SCS, warm_start=True, verbose=False)

    if w.value is None:
        return w_prev.copy()

    return np.asarray(w.value).reshape(-1)


def run_one_step(mu_series, Sigma_series, gamma=5.0, lam=0.06, w_max=0.10, ridge=1e-3):
    T, n = mu_series.shape
    w_prev = np.ones(n) / n
    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_one_step(
            mu_series[t], Sigma_series[t], w_prev,
            gamma=gamma, lam=lam, w_max=w_max, ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 8) Advanced convex optimiser
# ============================================================
def solve_advanced_one_step(
    mu,
    Sigma,
    w_prev,
    w_bench,
    F=None,
    gamma=5.0,
    eta=2.0,
    lam=0.05,
    kappa=0.10,
    w_max=0.10,
    ridge=1e-3
):
    n = len(mu)
    w = cp.Variable(n)

    Sigma_psd = make_psd(Sigma, ridge=ridge)

    turnover = cp.norm1(w - w_prev)
    abs_risk = cp.quad_form(w, Sigma_psd)
    te_risk = cp.quad_form(w - w_bench, Sigma_psd)

    vol = np.sqrt(np.maximum(np.diag(Sigma), 1e-10))
    D = np.diag(vol)
    robust_penalty = cp.norm2(D @ w)

    obj = cp.Maximize(
        mu @ w
        - gamma * abs_risk
        - eta * te_risk
        - lam * turnover
        - kappa * robust_penalty
    )

    cons = [
        cp.sum(w) == 1,
        w >= 0,
        w <= w_max
    ]

    if F is not None:
        cons.append(F.T @ (w - w_bench) == 0)

    prob = cp.Problem(obj, cons)

    try:
        prob.solve(solver=cp.SCS, warm_start=True, verbose=False)
    except Exception:
        return w_prev.copy()

    if w.value is None:
        return w_prev.copy()

    w_out = np.asarray(w.value).reshape(-1)
    w_out = np.maximum(w_out, 0)
    s = w_out.sum()
    if s > 0:
        w_out = w_out / s

    return w_out


def run_advanced_one_step(
    mu_series,
    Sigma_series,
    gamma=5.0,
    eta=2.0,
    lam=0.05,
    kappa=0.10,
    w_max=0.10,
    ridge=1e-3,
    F=None
):
    T, n = mu_series.shape
    w_prev = np.ones(n) / n
    w_bench = np.ones(n) / n
    W = np.zeros((T, n))

    for t in range(T):
        w_t = solve_advanced_one_step(
            mu=mu_series[t],
            Sigma=Sigma_series[t],
            w_prev=w_prev,
            w_bench=w_bench,
            F=F,
            gamma=gamma,
            eta=eta,
            lam=lam,
            kappa=kappa,
            w_max=w_max,
            ridge=ridge
        )
        W[t] = w_t
        w_prev = w_t

    return W


# ============================================================
# 9) Helpers: turnover, equity, stats
# ============================================================
def turnover_from_weights(W):
    return np.sum(np.abs(W[1:] - W[:-1]), axis=1)


def equity_from_weights_and_returns(R_daily, W, cost_rate=0.0005):
    """
    R_daily[t] is return from t-1 -> t.
    Use W[t-1] for R_daily[t] (causal).
    Transaction costs charged on turnover from W[t-1] -> W[t].
    """
    rp_gross = np.sum(R_daily[1:] * W[:-1], axis=1)
    to = turnover_from_weights(W)
    rp_net = rp_gross - cost_rate * to
    eq_gross = np.cumprod(1.0 + rp_gross)
    eq_net = np.cumprod(1.0 + rp_net)
    return eq_gross, eq_net, rp_gross, rp_net, to


def max_drawdown(equity_curve):
    peak = np.maximum.accumulate(equity_curve)
    dd = equity_curve / peak - 1.0
    return np.min(dd)


def annualized_sharpe(rp, periods_per_year=252):
    rp = np.asarray(rp)
    vol = np.std(rp)
    if vol < 1e-12:
        return np.nan
    return np.sqrt(periods_per_year) * np.mean(rp) / vol


def summarise_strategy(name, W, R_daily, cost_rate):
    eq_g, eq_n, rp_g, rp_n, to = equity_from_weights_and_returns(R_daily, W, cost_rate=cost_rate)
    return {
        "name": name,
        "final_equity_gross": eq_g[-1],
        "final_equity_net": eq_n[-1],
        "sharpe_gross": annualized_sharpe(rp_g),
        "sharpe_net": annualized_sharpe(rp_n),
        "max_dd_gross": max_drawdown(eq_g),
        "max_dd_net": max_drawdown(eq_n),
        "avg_turnover": np.mean(to),
        "avg_max_weight": np.mean(np.max(W, axis=1)),
        "has_nan_weights": bool(np.isnan(W).any()),
        "eq_gross": eq_g,
        "eq_net": eq_n,
        "rp_gross": rp_g,
        "rp_net": rp_n,
        "turnover": to,
    }


# ============================================================
# 10) CSV save helpers
# ============================================================
def ensure_dir(path):
    os.makedirs(path, exist_ok=True)


def save_chosen_assets_csv(seed_dir, seed, chosen_assets):
    df = pd.DataFrame({
        "seed": seed,
        "asset": chosen_assets
    })
    df.to_csv(os.path.join(seed_dir, f"chosen_assets_seed_{seed}.csv"), index=False)


def save_summary_csv(seed_dir, seed, results):
    rows = []
    for r in results:
        rows.append({
            "seed": seed,
            "strategy": r["name"],
            "final_equity_gross": r["final_equity_gross"],
            "final_equity_net": r["final_equity_net"],
            "sharpe_gross": r["sharpe_gross"],
            "sharpe_net": r["sharpe_net"],
            "max_dd_gross": r["max_dd_gross"],
            "max_dd_net": r["max_dd_net"],
            "avg_turnover": r["avg_turnover"],
            "avg_max_weight": r["avg_max_weight"],
            "has_nan_weights": r["has_nan_weights"],
        })
    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"summary_seed_{seed}.csv"), index=False)
    return df


def save_timeseries_csv(seed_dir, seed, times, results):
    rows = []
    time_index = pd.to_datetime(times[1:])  # because eq/turnover use R_daily[1:] and W[:-1]

    for r in results:
        for i, dt in enumerate(time_index):
            rows.append({
                "seed": seed,
                "date": dt,
                "strategy": r["name"],
                "eq_gross": r["eq_gross"][i],
                "eq_net": r["eq_net"][i],
                "rp_gross": r["rp_gross"][i],
                "rp_net": r["rp_net"][i],
                "turnover": r["turnover"][i],
            })

    df = pd.DataFrame(rows)
    df.to_csv(os.path.join(seed_dir, f"timeseries_seed_{seed}.csv"), index=False)
    return df


def save_weights_csv(seed_dir, seed, times, chosen_assets, weights_dict):
    weight_rows = []
    time_index = pd.to_datetime(times)

    for strategy_name, W in weights_dict.items():
        for t, dt in enumerate(time_index):
            row = {
                "seed": seed,
                "date": dt,
                "strategy": strategy_name
            }
            for j, asset in enumerate(chosen_assets):
                row[str(asset)] = W[t, j]
            weight_rows.append(row)

    df = pd.DataFrame(weight_rows)
    df.to_csv(os.path.join(seed_dir, f"weights_seed_{seed}.csv"), index=False)
    return df


def save_sanity_csv(seed_dir, seed, W_simple_base, W_ar1_base, W_simple_adv, W_ar1_adv):
    sanity = pd.DataFrame([{
        "seed": seed,
        "nan_simple_baseline": bool(np.isnan(W_simple_base).any()),
        "nan_ar1_baseline": bool(np.isnan(W_ar1_base).any()),
        "nan_simple_advanced": bool(np.isnan(W_simple_adv).any()),
        "nan_ar1_advanced": bool(np.isnan(W_ar1_adv).any()),
        "avg_max_weight_simple_baseline": float(np.mean(W_simple_base.max(axis=1))),
        "avg_max_weight_ar1_baseline": float(np.mean(W_ar1_base.max(axis=1))),
        "avg_max_weight_simple_advanced": float(np.mean(W_simple_adv.max(axis=1))),
        "avg_max_weight_ar1_advanced": float(np.mean(W_ar1_adv.max(axis=1))),
        "avg_turnover_simple_baseline": float(np.mean(turnover_from_weights(W_simple_base))),
        "avg_turnover_ar1_baseline": float(np.mean(turnover_from_weights(W_ar1_base))),
        "avg_turnover_simple_advanced": float(np.mean(turnover_from_weights(W_simple_adv))),
        "avg_turnover_ar1_advanced": float(np.mean(turnover_from_weights(W_ar1_adv))),
    }])
    sanity.to_csv(os.path.join(seed_dir, f"sanity_seed_{seed}.csv"), index=False)
    return sanity


# ============================================================
# 11) Single-seed run
# ============================================================
def run_single_seed(seed, data, out_dir, config):
    seed_dir = os.path.join(out_dir, f"seed_{seed}")
    ensure_dir(seed_dir)

    chosen_assets = load_quantiacs_universe(seed=seed, n_assets=config["n_assets"])[1]
    save_chosen_assets_csv(seed_dir, seed, chosen_assets)

    print(f"\nRunning seed {seed} | first 10 assets: {chosen_assets[:10]}")

    times, R_daily = compute_returns_matrix(data, chosen_assets, use_log_returns=False)

    Sigma_series = compute_sigma_series(
        R_daily,
        lookback=config["lookback_sigma"],
        annualize=config["annualize"]
    )

    mu_simple = mu_series_simple(
        R_daily,
        lookback_mu=config["K_simple"],
        mode="momentum",
        annualize=config["annualize"]
    )

    mu_ar1 = mu_series_ar1(
        R_daily,
        lookback_model=config["ar_window"],
        annualize=config["annualize"],
        smooth_alpha=config["smooth_alpha"]
    )

    W_simple_base = run_one_step(
        mu_simple, Sigma_series,
        gamma=config["gamma_base"],
        lam=config["lam_simple_base"],
        w_max=config["w_max"],
        ridge=config["ridge"]
    )

    W_ar1_base = run_one_step(
        mu_ar1, Sigma_series,
        gamma=config["gamma_base"],
        lam=config["lam_ar1_base"],
        w_max=config["w_max"],
        ridge=config["ridge"]
    )

    W_simple_adv = run_advanced_one_step(
        mu_simple, Sigma_series,
        gamma=config["gamma_adv"],
        eta=config["eta_adv"],
        lam=config["lam_simple_adv"],
        kappa=config["kappa_adv"],
        w_max=config["w_max"],
        ridge=config["ridge"],
        F=None
    )

    W_ar1_adv = run_advanced_one_step(
        mu_ar1, Sigma_series,
        gamma=config["gamma_adv"],
        eta=config["eta_adv"],
        lam=config["lam_ar1_adv"],
        kappa=config["kappa_adv"],
        w_max=config["w_max"],
        ridge=config["ridge"],
        F=None
    )

    save_sanity_csv(seed_dir, seed, W_simple_base, W_ar1_base, W_simple_adv, W_ar1_adv)

    results = []
    results.append(summarise_strategy("Simple + Baseline", W_simple_base, R_daily, config["cost_rate"]))
    results.append(summarise_strategy("AR1 + Baseline", W_ar1_base, R_daily, config["cost_rate"]))
    results.append(summarise_strategy("Simple + Advanced", W_simple_adv, R_daily, config["cost_rate"]))
    results.append(summarise_strategy("AR1 + Advanced", W_ar1_adv, R_daily, config["cost_rate"]))

    summary_df = save_summary_csv(seed_dir, seed, results)
    timeseries_df = save_timeseries_csv(seed_dir, seed, times, results)

    weights_dict = {
        "Simple + Baseline": W_simple_base,
        "AR1 + Baseline": W_ar1_base,
        "Simple + Advanced": W_simple_adv,
        "AR1 + Advanced": W_ar1_adv,
    }

    if config["save_weights"]:
        save_weights_csv(seed_dir, seed, times, chosen_assets, weights_dict)

    return {
        "seed": seed,
        "chosen_assets": chosen_assets,
        "summary_df": summary_df,
        "timeseries_df": timeseries_df,
        "weights_dict": weights_dict,
        "results": results,
    }


# ============================================================
# 12) Main batch run across 25 seeds
# ============================================================
if __name__ == "__main__":
    # ---------------- settings ----------------
    n_assets = 50
    n_seeds = 25
    seeds = list(range(1, n_seeds + 1))

    config = {
        "n_assets": n_assets,
        "lookback_sigma": 252,
        "K_simple": 60,
        "ar_window": 252,
        "annualize": 252,
        "gamma_base": 5.0,
        "lam_simple_base": 0.06,
        "lam_ar1_base": 0.20,
        "gamma_adv": 5.0,
        "eta_adv": 2.0,
        "lam_simple_adv": 0.06,
        "lam_ar1_adv": 0.20,
        "kappa_adv": 0.10,
        "w_max": 0.10,
        "ridge": 1e-3,
        "cost_rate": 0.0005,
        "smooth_alpha": 0.20,
        "save_weights": True,   # set False if files become too large
    }

    out_dir = "quantiacs_seed_runs"
    ensure_dir(out_dir)

    # ---------------- load Quantiacs data once ----------------
    print("Loading Quantiacs data once...")
    data = qndata.stocks.load_data()

    # ---------------- run all seeds ----------------
    all_summary = []
    all_timeseries = []

    for seed in seeds:
        try:
            run_out = run_single_seed(seed, data, out_dir, config)
            all_summary.append(run_out["summary_df"])
            all_timeseries.append(run_out["timeseries_df"])
        except Exception as e:
            print(f"Seed {seed} failed: {e}")
            fail_df = pd.DataFrame([{
                "seed": seed,
                "error": str(e)
            }])
            fail_df.to_csv(os.path.join(out_dir, f"seed_{seed}_FAILED.csv"), index=False)

    # ---------------- combine all seed results ----------------
    if len(all_summary) > 0:
        combined_summary = pd.concat(all_summary, ignore_index=True)
        combined_summary.to_csv(os.path.join(out_dir, "all_seeds_summary.csv"), index=False)

        avg_by_strategy = (
            combined_summary
            .groupby("strategy", as_index=False)
            .agg({
                "final_equity_gross": "mean",
                "final_equity_net": "mean",
                "sharpe_gross": "mean",
                "sharpe_net": "mean",
                "max_dd_gross": "mean",
                "max_dd_net": "mean",
                "avg_turnover": "mean",
                "avg_max_weight": "mean",
            })
        )
        avg_by_strategy.to_csv(os.path.join(out_dir, "average_metrics_across_seeds.csv"), index=False)

        print("\n=== Average metrics across seeds ===")
        print(avg_by_strategy)

    if len(all_timeseries) > 0:
        combined_timeseries = pd.concat(all_timeseries, ignore_index=True)
        combined_timeseries.to_csv(os.path.join(out_dir, "all_seeds_timeseries.csv"), index=False)

        avg_timeseries = (
            combined_timeseries
            .groupby(["date", "strategy"], as_index=False)
            .agg({
                "eq_gross": "mean",
                "eq_net": "mean",
                "rp_gross": "mean",
                "rp_net": "mean",
                "turnover": "mean",
            })
        )
        avg_timeseries.to_csv(os.path.join(out_dir, "average_timeseries_across_seeds.csv"), index=False)

        # ============================================================
        # 13) Optional plots from averages across seeds
        # ============================================================

        # Net equity averaged across seeds
        plt.figure(figsize=(10, 5))
        for strategy in avg_timeseries["strategy"].unique():
            tmp = avg_timeseries[avg_timeseries["strategy"] == strategy].sort_values("date")
            plt.plot(pd.to_datetime(tmp["date"]), tmp["eq_net"], label=strategy)
        plt.title("Average net equity across seeds")
        plt.xlabel("Date")
        plt.ylabel("Equity")
        plt.legend()
        plt.tight_layout()
        plt.show()

        # Turnover averaged across seeds
        plt.figure(figsize=(10, 5))
        for strategy in avg_timeseries["strategy"].unique():
            tmp = avg_timeseries[avg_timeseries["strategy"] == strategy].sort_values("date")
            plt.plot(pd.to_datetime(tmp["date"]), tmp["turnover"], label=strategy)
        plt.title("Average turnover across seeds")
        plt.xlabel("Date")
        plt.ylabel("Turnover")
        plt.legend()
        plt.tight_layout()
        plt.show()

    print(f"\nAll done. Results saved in folder: {out_dir}")